In [ ]:
import kagglehub
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import WeightedRandomSampler
from torch.utils.data import random_split
import torch
import numpy as np
from torchvision import datasets, transforms
from skopt.space import Real, Integer, Categorical

In [ ]:
# Download latest version
path = kagglehub.dataset_download("subhaditya/fer2013plus")

print("Path to dataset files:", path)

In [ ]:

data_transforms = {
    "train": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.RandomCrop(48, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ]),
    "test": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
}

# Paths to dataset
train_dir = path + "/fer2013plus/fer2013/train"
test_dir = path + "/fer2013plus/fer2013/test"

train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms["train"])
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms["test"])


In [ ]:
def create_dataloaders(train_dataset, test_dataset, batch_size, val_split=0.2):
    # Split train_dataset into training and validation datasets
    train_size = int((1 - val_split) * len(train_dataset))
    val_size = len(train_dataset) - train_size
    train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

    # Extract labels for the training subset
    train_labels = [train_dataset.targets[i] for i in train_subset.indices]

    # Calculate class counts and weights
    class_counts = np.bincount(train_labels)  # Assumes labels are sequential integers starting from 0
    class_weights = 1. / np.sqrt(class_counts + 1e-6)  # Avoid division by zero
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    # Create sample weights for WeightedRandomSampler
    sample_weights = [class_weights[label] for label in train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    # Create DataLoaders
    train_loader = DataLoader(train_subset, batch_size=batch_size, sampler=sampler)  # Stratified sampling for training
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)  # No sampling for validation
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)  # Random shuffle for test


    return train_loader, val_loader, test_loader


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(in_features=128 * 6 * 6, out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features=8)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion, num_epochs=100, print_every=100, patience=5):
    """
    Trains a given model with early stopping and specified parameters on GPU.

    Args:
        model: The model to train.
        train_loader: DataLoader for the training dataset.
        val_loader: DataLoader for the validation dataset.
        optimizer: Optimizer for training (e.g., Adam, SGD).
        criterion: Loss function (e.g., CrossEntropyLoss).
        num_epochs: Number of training epochs.
        print_every: Frequency (in batches) to print training loss.
        patience: Number of epochs to wait for validation loss improvement before stopping.
    """
    # Set device (CUDA if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Move the model to GPU (if available)
    model = model.to(device)

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()  # Set the model to training mode
        running_loss = 0.0

        for i, (inputs, labels) in enumerate(train_loader):
            # Move data to GPU (if available)
            inputs, labels = inputs.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Accumulate the loss
            running_loss += loss.item()

            # Print progress
            if (i + 1) % print_every == 0:
                print(
                    f"Epoch [{epoch + 1}/{num_epochs}], "
                    f"Step [{i + 1}/{len(train_loader)}], "
                    f"Loss: {running_loss / print_every:.4f}"
                )
                running_loss = 0.0

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                # Move data to GPU
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Validation Loss: {val_loss:.4f}")

        # Check for early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0  # Reset patience counter if improvement
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break


In [ ]:
def evaluate_model(model, data_loader):
    """
    Evaluates the model's performance on a given dataset.

    Args:
        model: The trained model to evaluate.
        data_loader: DataLoader for the dataset to evaluate.

    Returns:
        float: Accuracy of the model on the dataset.
    """
    # Set the device (CUDA if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Move model to the device
    model = model.to(device)

    model.eval()  # Set model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            # Move data to the same device as model
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    return accuracy


In [ ]:
def visualize_predictions(model, data_loader, classes, num_images=8, device='cpu'):
    """
    Visualizes original images with ground truth and predicted labels.

    Args:
        model: The trained model to use for predictions.
        data_loader: DataLoader for the dataset to evaluate.
        classes: List of class names corresponding to the labels.
        num_images: Number of images to visualize (default is 8).
        device: Device to use for model and data ('cpu' or 'cuda').

    """
    # Set model to evaluation mode
    model.eval()
    model.to(device)

    # Get a random batch of data
    dataiter = iter(data_loader)
    images, labels = next(dataiter)
    images, labels = images.to(device), labels.to(device)

    # Get predictions from the model
    with torch.no_grad():
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

    # Limit the number of images to visualize
    num_images = min(num_images, len(images))

    # Create a grid of images
    rows = (num_images + 1) // 2  # Number of rows for layout
    cols = 2  # Fixed number of columns

    plt.figure(figsize=(15, rows * 5))  # Adjust figure size
    for i in range(num_images):
        ax = plt.subplot(rows, cols, i + 1)

        # Convert tensor to numpy array for display
        img = images[i].cpu().permute(1, 2, 0).numpy()  # Rearrange dimensions for matplotlib

        plt.imshow(img)  # Display the original image

        # Add ground truth and predicted labels
        true_label = classes[labels[i].item()]
        pred_label = classes[predicted[i].item()]
        ax.set_title(f"True: {true_label}\nPred: {pred_label}", fontsize=12)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
search_space = [
    Real(1e-4, 1e-2, name='learning_rate', prior='log-uniform'),
    Integer(16, 32, name='batch_size'),  # Smaller batch size range
    Categorical(['Adam'], name='optimizer'),  # Fixed optimizer
    Integer(3, 5, name='patience')  # Early stopping patience
]


In [ ]:
def objective(params):
    learning_rate, batch_size, optimizer_type, patience = params
    batch_size = int(batch_size)
    patience = int(patience)

    # Set the device (use GPU if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Update data loaders
    train_loader, val_loader, _ = create_dataloaders(train_dataset, test_dataset, batch_size)

    # Define model, criterion, and optimizer
    model = CNN().to(device)  # Move model to GPU
    criterion = nn.CrossEntropyLoss()

    # Use the correct optimizer from torch.optim
    optimizer = getattr(optim, optimizer_type)(model.parameters(), lr=learning_rate)

    # Early stopping logic
    best_val_loss = float('inf')
    patience_counter = 0

    print(
        f"Starting training with params: lr={learning_rate}, batch_size={batch_size}, optimizer={optimizer_type}, patience={patience}, device={device}")

    for epoch in range(25):  # Use a high fixed max epoch value
        model.train()
        running_loss = 0.0

        # Training loop
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            # Print batch progress
            if (batch_idx + 1) % 100 == 0:  # Every 100 batches
                print(f"Epoch [{epoch + 1}/25], Batch [{batch_idx + 1}/{len(train_loader)}], Loss: {loss.item():.4f}")

        # Validation loss
        model.eval()
        val_loss = 0.0
        with torch.no_grad():  # Use torch.no_grad() instead of torchvision.no_grad()
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        print(f"Epoch [{epoch + 1}/25], Validation Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            print(f"Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}")
            best_val_loss = val_loss
            patience_counter = 0  # Reset patience counter if improvement
        else:
            patience_counter += 1
            print(f"No improvement in validation loss. Patience counter: {patience_counter}/{patience}")

            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch + 1} epochs")
                break

    # Calculate validation accuracy
    total, correct = 0, 0
    with torch.no_grad():  # Use torch.no_grad() instead of torchvision.no_grad()
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)  # Use torch.max() instead of torchvision.max()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = correct / total
    print(f"Final Validation Accuracy: {val_accuracy:.4f}")
    return -val_accuracy


In [ ]:
result = gp_minimize(
    func=objective,
    dimensions=search_space,
    n_calls=10,  # Reduced iterations
    random_state=42
)

print("Best parameters:", result.x)
print("Best validation accuracy:", -result.fun)


In [ ]:
# Extract best parameters from tuning results
#Best parameters: 0.0001930783753654713, np.int64(26), 'Adam', np.int64(4)
best_learning_rate = result.x[0]
best_batch_size = int(result.x[1])
best_optimizer_type = result.x[2]
best_patience = int(result.x[3])

# Update data loaders with the best batch size
train_loader, val_loader, test_loader = create_dataloaders(train_dataset, test_dataset, best_batch_size)

# Define the model, criterion, and optimizer
best_cnn_model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = getattr(torch.optim, best_optimizer_type)(best_cnn_model.parameters(), lr=best_learning_rate)

# Train the model with early stopping
train_model(
    model=best_cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=50,  # High fixed value, let early stopping decide
    print_every=100,
    patience=best_patience
)


In [ ]:
torch.save(best_cnn_model.state_dict(), 'best_cnn_model.pth')


In [ ]:
loaded_cnn_model = CNN()
loaded_cnn_model.load_state_dict(torch.load("best_cnn_model.pth", weights_only=True))

In [ ]:
test_accuracy = evaluate_model(loaded_cnn_model, test_loader)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
train_accuracy = evaluate_model(loaded_cnn_model, train_loader)
print(f"Train Accuracy: {train_accuracy * 100:.2f}%")

In [ ]:
# Prepare to count predictions for each class
classes = train_dataset.classes  # Get class names from the dataset
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# Set the device (CUDA if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Move the model to the selected device
cnn_model = loaded_cnn_model.to(device)

# Disable gradients for evaluation
with torch.no_grad():
    for data in test_loader:
        images, labels = data

        # Move data to the same device as the model
        images, labels = images.to(device), labels.to(device)

        # Get predictions from the model
        outputs = cnn_model(images)
        _, predictions = torch.max(outputs, 1)  # Get predicted class indices

        # Collect the correct predictions for each class
        for label, prediction in zip(labels, predictions):
            label_name = classes[label.item()]  # Convert label index to class name
            if label == prediction:
                correct_pred[label_name] += 1
            total_pred[label_name] += 1

# Print accuracy for each class
for classname, correct_count in correct_pred.items():
    if total_pred[classname] > 0:  # Avoid division by zero
        accuracy = 100 * float(correct_count) / total_pred[classname]
        print(f'Accuracy for class: {classname:10s} is {accuracy:.1f} %')
    else:
        print(f'No predictions for class: {classname}')


In [ ]:
# Define class names
classes = train_dataset.classes  # Assuming you use ImageFolder or a similar dataset

# Visualize predictions
visualize_predictions(
    model=loaded_cnn_model,  # Your trained model
    data_loader=test_loader,  # DataLoader for the test dataset
    classes=classes,  # Class names
    num_images=8,  # Number of images to visualize
    device='cpu'  # Use 'cuda' if running on GPU
)